# End-to-end agent pipeline on Colab

**Architecture**: Setup → Outline → for each chapter [Retriever (RAG) → Writer (LoRA) → Checker → rewrite up to 3 times]

**Prerequisites**: under Drive `tunshi_lora/` you must have:
- `checkpoints/final/` — the LoRA adapter
- `vectordb/` — the RAG vector store

**Expected time**: ~8 minutes to load + 30-50 minutes to generate three chapters (with up to nine rewrites).

## Cell 1 — Mount Drive and verify prerequisites

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = '/content/drive/MyDrive/tunshi_lora/checkpoints/final'
VECTORDB_SRC = '/content/drive/MyDrive/tunshi_lora/vectordb'
VECTORDB_LOCAL = '/content/vectordb'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'), f'❌ LoRA adapter 缺失: {ADAPTER_DIR}'
assert os.path.exists(f'{VECTORDB_SRC}/chroma.sqlite3'), f'❌ vectordb 没上传到 Drive: {VECTORDB_SRC}'

print('✅ Adapter:')
!ls -lh {ADAPTER_DIR}/adapter_model.safetensors

print('\n✅ vectordb (Drive):')
!du -sh {VECTORDB_SRC}

# Copy vectordb to Colab local disk (Drive direct reads are slow)
if not os.path.exists(VECTORDB_LOCAL):
    print(f'\n复制 vectordb 到 Colab 本地（30 秒）...')
    !cp -r {VECTORDB_SRC} {VECTORDB_LOCAL}
print(f'\n✅ vectordb 在 {VECTORDB_LOCAL}')
!ls {VECTORDB_LOCAL}

## Cell 2 — Install dependencies

In [2]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.0.1 sentence-transformers chromadb
!pip uninstall -y bitsandbytes 2>/dev/null

import transformers, peft, sentence_transformers, chromadb
print(f'transformers:          {transformers.__version__}')
print(f'peft:                  {peft.__version__}')
print(f'sentence-transformers: {sentence_transformers.__version__}')
print(f'chromadb:              {chromadb.__version__}')

## Cell 3 — Load the LoRA-adapted model (5-8 minutes on first run)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'Qwen/Qwen2.5-7B'

print('[1/3] 加载 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('[2/3] 加载 base Qwen2.5-7B (bf16) ...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
).to('cuda')

print('[3/3] 套 LoRA adapter ...')
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print(f'\n✅ 显存: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

## Cell 4 — Load the RAG components (BGE-zh + ChromaDB)

In [4]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

print('[1/2] Loading BAAI/bge-base-zh-v1.5 (first download ~390 MB) ...')
bge = SentenceTransformer('BAAI/bge-base-zh-v1.5', device='cuda')

print('[2/2] Connecting to ChromaDB ...')
client = chromadb.PersistentClient(
    path=VECTORDB_LOCAL,
    settings=Settings(anonymized_telemetry=False))
collection = client.get_collection('tunshi_worldview_v1')
print(f'✅ collection.count() = {collection.count()}')


def rag_search_single(query, top_k=8, source_filter=None, dist_max=0.55):
    """One query → de-duplicated hits below the distance threshold."""
    emb = bge.encode(query, normalize_embeddings=True, convert_to_numpy=True).tolist()
    where = {'source': source_filter} if source_filter else None
    results = collection.query(
        query_embeddings=[emb], n_results=top_k, where=where,
        include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        if dist > dist_max:
            continue
        hits.append({'text': doc, 'source': meta['source'],
                     'raw_line': meta['raw_line'], 'dist': dist})
    return hits


def rag_search(queries, top_k=8, source_filter=None, dist_max=0.55, total_max=20):
    """Multi-query aggregation: run each query, merge, de-dupe by raw_line,
    sort by distance, cap at total_max."""
    if isinstance(queries, str):
        queries = [queries]
    seen = set()
    all_hits = []
    for q in queries:
        for h in rag_search_single(q, top_k=top_k, source_filter=source_filter, dist_max=dist_max):
            key = (h['source'], h['raw_line'])
            if key in seen:
                continue
            seen.add(key)
            all_hits.append(h)
    all_hits.sort(key=lambda x: x['dist'])
    return all_hits[:total_max]


# Smoke test
print('\n=== RAG smoke test ===')
test_hits = rag_search(['恒星级 修炼', '武者突破'], top_k=4, total_max=5)
for h in test_hits:
    print(f"  [{h['source']} 行{h['raw_line']} dist={h['dist']:.2f}] {h['text'][:80]}...")


## Cell 5 — Setup, generation function, helpers

In [5]:
# === Setup ===
# All fields are user-editable. The book title and chapter titles
# are not hardcoded here — they are generated by the Outline agent
# in Cell 7 from this Setup.
SETUP = {
    'protagonist_name': '风灵',
    'time_anchor': '比罗峰出生早5年',
    'place_anchor': '地球',
    'tier_anchor': '没有限制',
    'golden_finger': '某半浑源传承，一块儿黑色石头',
}

# === LLM generation function ===
def generate(prompt, max_new_tokens=1500, temperature=0.85, top_p=0.9, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3500).to('cuda')
    prompt_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=top_p,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

print('✅ Setup + generate() ready')
for k, v in SETUP.items():
    print(f'  {k}: {v}')


## Cell 6 — Checker (seven hard rules with automatic scoring)


In [6]:
import re

# AI fragmentation regex: 3+ consecutive period-terminated lines of <=8 chars
AI_SHORT_RE = re.compile(r'(^[^。\n]{1,8}。\n){3,}', re.MULTILINE)

# Web-novel meta-noise: residue from the LoRA training corpus
#   author notes, reader comments, serialisation markers, solicitation, forum speaker tags, etc.
NOISE_PATTERNS = [
    r'未完待续', r'还在连载', r'连载中', r'下一章',
    r'求收藏', r'求推荐', r'求订阅', r'求月票', r'求打赏',
    r'感谢.{0,10}打赏', r'感谢.{0,10}书友', r'感谢.{0,10}读者',
    r'作者的话', r'作者注', r'读者.{0,5}说', r'读者评论',
    r'本章完', r'章节内容', r'书友们', r'本书读者群',
    r'第\s*\d+\s*章\s*结束', r'第\s*\d+\s*章\s*已[经开]',
    r'^\s*[A-Z]\s*[:：]\s',  # forum speaker tags (uses re.MULTILINE)
    r'ixdzs', r'爱下电', r'txt80',
]
NOISE_RE = re.compile('|'.join(NOISE_PATTERNS), re.MULTILINE)

# Placeholder residue: model echoing back '[XX]' / '【XX】' template scaffolding
# (e.g. '【主角】风灵', '[书名]', '【时空】')
PLACEHOLDER_RE = re.compile(r'[【\[][\u4e00-\u9fff]{1,8}[】\]]')


def checker(text, setup):
    """Seven hard rules. Returns (passed, issues_list, score)."""
    issues = []

    # 1. AI-style fragmentation
    m = AI_SHORT_RE.search(text)
    if m:
        issues.append(f"Rule 1 (AI fragmentation): 3+ short period-terminated lines at pos {m.start()}")

    # 2. Protagonist name presence
    name = setup['protagonist_name']
    name_count = text.count(name)
    if name_count < 5:
        issues.append(f"Rule 2 (protagonist): '{name}' appears only {name_count} times (< 5)")

    # 3. Web-novel paragraph format
    fw_pairs = text.count('　　')
    lines = text.count('\n')
    ratio = fw_pairs / max(1, lines)
    if ratio < 0.2:
        issues.append(f"Rule 3 (paragraph format): 　　 ratio {ratio:.2f} < 0.2")

    # 4. Chapter length
    if len(text) < 1200:
        issues.append(f"Rule 4 (length): {len(text)} chars < 1200")

    # 5. Meta-noise residue (training corpus leakage)
    noise_hits = NOISE_RE.findall(text)
    if noise_hits:
        unique = list(dict.fromkeys(noise_hits))[:5]
        issues.append(f"Rule 5 (meta-noise): leaked markers: {unique}")

    # 6. Placeholder residue (model echoing template scaffolding)
    ph_hits = PLACEHOLDER_RE.findall(text)
    if ph_hits:
        unique = list(dict.fromkeys(ph_hits))[:5]
        issues.append(f"Rule 6 (placeholder echo): {unique}")

    # 7. Golden finger embodiment
    gf = setup.get('golden_finger', '')
    gf_keywords = re.findall(r'[\u4e00-\u9fff]{2,4}', gf)
    STOP = {'一块', '神秘', '某个', '一种', '一枚', '一颗', '一份', '一团',
            '能在', '可以', '可能', '比如', '或者', '什么', '怎么样',
            '突然', '不能', '不会'}
    gf_keywords = [k for k in gf_keywords if k not in STOP][:6]
    if gf_keywords:
        if not any(k in text for k in gf_keywords):
            issues.append(f"Rule 7 (golden-finger embodiment): none of {gf_keywords} appears in this chapter")

    score = max(0, 100 - len(issues) * 14)
    return len(issues) == 0, issues, score


# self-test
test_text = '　　风灵站在街头。\n　　风灵深吸一口气。\n风灵笑了。\n他不笑。\n他不哭。'
passed, issues, score = checker(test_text, {'protagonist_name': '风灵', 'golden_finger': '一块黑色石头'})
print(f'Checker self-test: passed={passed}, score={score}')
for i in issues: print(f'  ❌ {i}')


## Cell 7 — Setup agent + Outline agent

In [7]:
import time, re

# === RAG: multi-angle retrieval (covers tier, golden finger, place, era) ===
rag_queries = [
    f'{SETUP["tier_anchor"]} 修炼 突破',
    f'{SETUP["golden_finger"]}',
    f'{SETUP["place_anchor"]}',
    f'{SETUP["time_anchor"]} 历史',
    f'{SETUP["protagonist_name"]} 主角',
    '武者 境界 体系',
]
hits = rag_search(rag_queries, top_k=6, total_max=24)
rag_context = '\n'.join(
    f"- [{h['source']} 行{h['raw_line']}] {h['text'][:200]}" for h in hits[:18]
)
print(f'=== RAG retrieved {len(hits)} unique chunks across {len(rag_queries)} queries ===')
print(rag_context[:600])
print('...')

# === Outline generation ===
# IMPORTANT: do NOT put bracketed placeholders like [书名] / [章节名] inside the
# template — the model will copy them verbatim. We use trailing-prompt
# continuation instead: end with "书名：《" so the model fills in directly.

print('\n=== Generating outline (book title + 3 chapter titles + plot beats) ===')

outline_prompt = f"""请为《吞噬星空》衍生宇宙写一部 3 章原创小说大纲。

【主角设定（必须严格遵守）】
- 主角姓名：{SETUP["protagonist_name"]}
- 起始境界：{SETUP["tier_anchor"]}
- 时空背景：{SETUP["time_anchor"]} · {SETUP["place_anchor"]}
- 金手指（核心驱动）：{SETUP["golden_finger"]}

【创作要求】
1. 三章必须围绕「金手指」展开：第1章首次激活/感知它，第2章用它解决一个具体冲突，第3章因它招致更大格局。
2. 章节名 2-4 个中文字，体现各章主题，不要叫《序章》《楔子》。
3. 书名 4-8 个中文字，反映「{SETUP["protagonist_name"]}」与金手指的关系，不可与已有原著重名。
4. 每章 4 个情节点：场景 + 冲突 + 主角动作 + 章末小钩子。
5. 主角周围出现的角色全部是你新造的人物，不要套用原著角色名。

【参考世界观（来自 RAG 检索的注解，含具体行号引用）】
{rag_context[:2000]}

---

请直接输出，不要任何前言。开始：

书名：《"""

t0 = time.time()
outline_body = generate(outline_prompt, max_new_tokens=1400, temperature=0.75)
outline = '书名：《' + outline_body  # prepend the continuation seed for parsing
print(outline)
print(f'\n[took {time.time()-t0:.1f}s]')

# === Parse book title and chapter titles ===
m_book = re.search(r'书名[:：]\s*《\s*([^》\n]+?)\s*》', outline)
book_title = m_book.group(1).strip() if m_book else f'{SETUP["protagonist_name"]}传'

m_chapters = re.findall(r'第\s*[1-3一二三]\s*章\s+([^\n]+)', outline)
chapter_titles = []
for raw in m_chapters[:3]:
    clean = re.split(r'情节点|[:：]|\s{2,}', raw, maxsplit=1)[0].strip()
    chapter_titles.append(clean if clean else f'第{len(chapter_titles)+1}章')
while len(chapter_titles) < 3:
    chapter_titles.append(f'第{len(chapter_titles)+1}章')

# Defend against placeholder residue 【XX】 [XX] in the parsed values
def strip_placeholder(s):
    return re.sub(r'[【\[][^】\]]{0,8}[】\]]', '', s).strip() or s
book_title = strip_placeholder(book_title)
chapter_titles = [strip_placeholder(t) or f'第{i+1}章' for i, t in enumerate(chapter_titles)]

BOOK_META = f'<book_meta>类别=D1 书名={book_title} 作者=AI</book_meta>'

print(f'\n=== Parsed from outline ===')
print(f'  Book title:     《{book_title}》')
print(f'  Chapter titles: {chapter_titles}')
print(f'  BOOK_META used by Writer: {BOOK_META}')


## Cell 8 — Writer + Checker rewrite loop (adversarial core)

In [8]:
MAX_RETRIES = 3


def write_chapter_with_retry(chapter_idx, chapter_title, prev_chapter_tail='', max_retries=MAX_RETRIES):
    """Generate one chapter + Checker verification + rewrite on failure.

    Returns (best_text, attempts_log).
    """
    # Multi-query RAG for this specific chapter
    ch_queries = [
        f'{SETUP["tier_anchor"]} 战斗',
        f'{SETUP["golden_finger"]}',
        f'{chapter_title}',
        f'{SETUP["place_anchor"]}',
    ]
    hits = rag_search(ch_queries, top_k=4, total_max=10)
    ch_rag = '\n'.join(
        f"- [{h['source']} 行{h['raw_line']}] {h['text'][:150]}" for h in hits[:8]
    )

    best = {'text': '', 'score': -1, 'issues': []}
    attempts_log = []

    for attempt in range(1, max_retries + 1):
        feedback = ''
        if attempt > 1 and best['issues']:
            feedback = (
                '\n【上次写作存在以下问题，本次必须避免】\n'
                + '\n'.join('  - ' + i for i in best['issues'])
                + '\n'
            )

        prompt = f"""{BOOK_META}
<chapter_title>第{chapter_idx}章 {chapter_title}</chapter_title>

【主角与设定（不可违背）】
- 主角：{SETUP["protagonist_name"]}，{SETUP["tier_anchor"]}
- 时空：{SETUP["time_anchor"]} · {SETUP["place_anchor"]}
- 金手指：{SETUP["golden_finger"]}（**本章必须有具体情节体现这个金手指**）

【参考世界观（仅用作背景常识，不要照搬人物名）】
{ch_rag}
{feedback}
【前章末尾衔接】
{prev_chapter_tail[-300:] if prev_chapter_tail else '（本章是开篇）'}

【写作要求】
- 番茄式短段落（中文全角双空格缩进，对话独立成段），约 2500 中文字。
- 主角名「{SETUP["protagonist_name"]}」至少出现 5 次。
- 本章必须围绕「金手指」展开至少一段具体描写。
- 不要出现「未完待续」「下一章」「作者的话」「求收藏」「读者评论」等同人文套话。
- 不要使用方括号或全角括号包裹的占位符（如「【主角】」「[书名]」）。

直接开始写正文，不要给前言或总结：

　　"""

        temp = 0.85 + 0.05 * (attempt - 1)
        seed = 42 + attempt

        t0 = time.time()
        text = generate(prompt, max_new_tokens=1800, temperature=temp, seed=seed)
        elapsed = time.time() - t0

        passed, issues, score = checker(text, SETUP)

        attempts_log.append({
            'attempt': attempt, 'temperature': temp, 'seed': seed,
            'score': score, 'issues': issues, 'elapsed': elapsed, 'length': len(text),
        })

        print(f'  [Attempt {attempt}] temp={temp:.2f}, score={score}/100, {len(text)} chars, {elapsed:.0f}s')
        for i in issues:
            print(f'    ❌ {i}')

        if score > best['score']:
            best = {'text': text, 'score': score, 'issues': issues}

        if passed:
            print(f'  ✅ Attempt {attempt} passed!')
            break

    if best['score'] < 100:
        print(f'  ⚠️ Chapter {chapter_idx} kept best-of-{max_retries} with {len(best["issues"])} residual issues')
    return best['text'], attempts_log


print('✅ write_chapter_with_retry() ready')


## Cell 9 — Run end to end: generate three chapters

**Expected time: 30-50 minutes** (up to 3 rewrites per chapter, ~50 s per attempt).

In [9]:
chapters = []
all_logs = []
prev_tail = ''

# chapter_titles was produced by the Outline agent in Cell 7.
# We do not hardcode them.
assert 'chapter_titles' in globals() and len(chapter_titles) == 3, \
    'chapter_titles must come from Cell 7; re-run Cell 7 first.'

global_t0 = time.time()

for idx, title in enumerate(chapter_titles, 1):
    print(f'\n{"=" * 70}')
    print(f'>>> Chapter {idx}: 《{title}》')
    print(f'{"=" * 70}')
    text, log = write_chapter_with_retry(idx, title, prev_tail)
    chapters.append({'idx': idx, 'title': title, 'text': text, 'log': log})
    all_logs.append(log)
    prev_tail = text[-500:]

total_elapsed = time.time() - global_t0
print(f'\n{"=" * 70}')
print(f'All 3 chapters generated. Total time: {total_elapsed/60:.1f} min')
print(f'{"=" * 70}')


## Cell 10 — Print the three chapters

In [10]:
for c in chapters:
    print('=' * 70)
    print(f'第{c["idx"]}章 《{c["title"]}》  ({len(c["text"])} 字)')
    print('=' * 70)
    print(f'　　{c["text"]}')
    print()

## Cell 11 — Final report and save to Drive

Run this after Cell 9 finishes. The LoRA adapter is ~200-250 MB; the outputs are written to Drive under `tunshi_lora/inference_runs/run_001/` and can be downloaded from there.

In [11]:
import json, os

# === Final report ===
print('=' * 70)
print('Adversarial Checker — final report')
print('=' * 70)
print(f'Book title: 《{book_title}》  (generated by the Outline agent)\n')
for c in chapters:
    log = c['log']
    final_score = log[-1]['score']
    n_attempts = len(log)
    n_issues = len(log[-1]['issues'])
    print(f'Chapter {c["idx"]} 《{c["title"]}》:')
    print(f'  attempts: {n_attempts}/3')
    print(f'  final score: {final_score}/100')
    print(f'  remaining issues: {n_issues}')
    if log[-1]['issues']:
        for i in log[-1]['issues']:
            print(f'    - {i}')
    print()

# === Save outputs to Drive ===
out_dir = '/content/drive/MyDrive/tunshi_lora/inference_runs/run_001'
os.makedirs(out_dir, exist_ok=True)

with open(f'{out_dir}/setup.json', 'w') as f:
    json.dump(SETUP, f, ensure_ascii=False, indent=2)
with open(f'{out_dir}/outline.txt', 'w') as f:
    f.write(f'书名：{book_title}\n\n{outline}')
for c in chapters:
    with open(f'{out_dir}/chapter_{c["idx"]}_{c["title"]}.txt', 'w') as f:
        f.write(c['text'])
with open(f'{out_dir}/check_log.json', 'w') as f:
    json.dump({
        'book_title': book_title,
        'setup': SETUP,
        'chapter_logs': [c['log'] for c in chapters],
    }, f, ensure_ascii=False, indent=2)

print(f'✅ saved to: {out_dir}')
!ls -lh {out_dir}/
